# From a policy to training data in five minutes

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Juneo-Lab/decision-forge/blob/main/examples/agent_delegation/tutorial.ipynb)

This notebook turns one organization's written agent policy into verified decision training data, changes the policy, and watches the labels follow. It runs offline: no API keys and no model downloads. The [tutorial page](https://github.com/Juneo-Lab/decision-forge/blob/main/docs/tutorial.md) walks through the same steps with the command line.

In [ ]:
# Use the repository this notebook lives in; in Colab, fetch and install it first.
import os
import subprocess
import sys
from pathlib import Path

root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "examples/agent_delegation").is_dir()), None)
if root is None:
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Juneo-Lab/decision-forge"], check=True)
    root = Path("decision-forge").resolve()
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", str(root)], check=True)
os.chdir(root)
print("Working in", root)

## 1. Start from a written policy

We copy the fictional Halcyon Health organization from the agent delegation example. Its policy is plain text written by the organization; one sentence governs refunds: *Agents may issue refunds of up to 100 dollars to verified patients without approval.*

In [ ]:
import json
import runpy
import shutil

from decision_forge.exporters import export_run
from decision_forge.pipeline import build, load_run

work = Path("runs/tutorial")
shutil.rmtree(work, ignore_errors=True)
shutil.copytree("examples/agent_delegation/halcyon-health", work / "my-org")
spec = work / "my-org/project.yaml"
policy = work / "my-org/sources/policy.md"
print(policy.read_text())

## 2. Build verified decisions

The specification declares what an agent may attempt, the situations to generate, and rules that each quote the sentence they encode. The build labels every situation by replaying those rules.

In [ ]:
report = build(spec, work / "v1")["report"]
print(report["accepted"], "situations by split:", report["by_split"])


def refund_label(run):
    """Label and split of the 400-dollar refund to a verified patient."""
    episode = next(
        e for e in load_run(run) if e.node_id == "refund" and e.state == {"amount": 400, "customer_verified": True}
    )
    return episode.valid_choices, episode.split


print("A 400-dollar refund to a verified patient:", refund_label(work / "v1"))

## 3. Change the policy, watch the labels follow

Raise the refund threshold to 500 dollars in the policy only. The build refuses, because a rule may not cite text the organization no longer says.

In [ ]:
policy.write_text(policy.read_text().replace("refunds of up to 100 dollars", "refunds of up to 500 dollars"))
try:
    build(spec, work / "v2")
except ValueError as error:
    print("Refused:", error)

Update the rule to match, both its quote and its threshold, and build again. The label changes; the split does not, because the first build recorded every family's split in `project.splits.lock.json`.

In [ ]:
spec.write_text(
    spec.read_text()
    .replace("refunds of up to 100 dollars", "refunds of up to 500 dollars")
    .replace("value: 100\n", "value: 500\n")
)
build(spec, work / "v2")
print("Before:", refund_label(work / "v1"))
print("After: ", refund_label(work / "v2"))

## 4. Export training data

The `decision` format gives each row explicit options, aligned target probabilities, a family for grouping, and provenance.

In [ ]:
print(export_run(work / "v2", work / "train", "decision"))
row = json.loads((work / "train/train.jsonl").read_text().splitlines()[0])
print(json.dumps({key: row[key] for key in ("input", "target_probs", "provenance")}, indent=2))

## 5. Compare two organizations

The same 17 situations, decided by a speed-first startup and by a clinic network. Every decision cites the sentence that governs it.

In [ ]:
compare = runpy.run_path("scripts/compare_organizations.py")
specs = [Path("examples/agent_delegation") / name / "project.yaml" for name in ("lumen-labs", "halcyon-health")]
markdown = compare["render_markdown"](compare["compare"](specs, work / "compare"))
try:
    from IPython.display import Markdown, display

    display(Markdown(markdown))
except ImportError:
    print(markdown)

## Next steps

- Write your own policy and specification, or let `compile-corpus` draft one from your documents through a model endpoint you choose; compiled specifications pass a content-bound business review before they build.
- Read [Why Decision Forge](https://github.com/Juneo-Lab/decision-forge/blob/main/docs/vision.md) for the problem this data is meant to solve.